# 01 — The widest street: the maximum margin

*Notebook 1 of 5 · Support Vector Machines*

Welcome to support vector machines. Logistic regression (Chapter 03) drew one separating line and
read a probability off it. But when two classes can be cleanly separated, *many* lines separate them
perfectly — and logistic regression never asks which line is best placed. Here you meet a different,
geometric answer: among all the lines that separate the classes, pick the one with the **widest empty
street** between them. You will build that idea by hand before calling the library once.

**Prerequisites**
- Chapter 03, NB 2 — the linear decision boundary $w \cdot x + b = 0$, and that the weight vector
  $w$ points perpendicular to it.
- Module 00, NB 04 — the train/test split; NB 05 — the nearest-centroid classifier (a single
  straight boundary), our visual foil.

**What you'll be able to do**
- Explain what the *margin* is, and why the widest one is the most defensible boundary.
- Find the *support vectors* and the maximum-margin boundary **by hand** on separable data.
- State why the margin width equals $2 / \lVert w \rVert$.
- Confirm that a linear `SVC` finds exactly the same street.
- Explain why only the support vectors matter — and nothing else does.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from ml_course import colors, viz

viz.use_course_style()
SEED = 0
np.random.seed(SEED)

# A small, cleanly separable two-class set: few enough points to reason about by hand.
X_raw, y = make_blobs(
    n_samples=40,
    centers=[[-2.2, -2.2], [2.2, 2.2]],
    cluster_std=0.7,
    random_state=SEED,
)
# Standardize: an SVM measures distances, so the feature scales must be comparable
# (why this matters is paid off in NB 5; here we put both features on an equal footing).
X = StandardScaler().fit_transform(X_raw)

separable_acc = SVC(kernel="linear", C=1e6).fit(X, y).score(X, y)
print(f"X shape: {X.shape}    class counts: {np.bincount(y)}")
print(f"hard-margin linear SVM accuracy on its own data: {separable_acc:.3f}")
print("(1.000 means the two classes are cleanly separable)")

## A line, and the question logistic regression left open

A linear classifier splits the plane with a straight boundary

$$ w \cdot x + b = 0, $$

calling one class where $w \cdot x + b > 0$ and the other where it is $< 0$. The weight vector $w$
points **perpendicular** to that boundary (Chapter 03).

On data that can be separated, a new question appears: **many** different lines classify every point
correctly. Logistic regression chose one by maximizing a likelihood. A support vector machine chooses
one by a purely geometric rule — and that rule turns out to generalize remarkably well.

## Many lines separate; which is best?

Picture several straight lines, each keeping every point of one class on its own side. They are not
equally trustworthy. A line that grazes the edge of one cloud is fragile: a single new point, measured
a hair differently, could land on the wrong side. A line with generous room on **both** sides is
robust.

The support vector machine makes this precise. The **margin** of a boundary is its distance to the
nearest point on each side; the **street** is the band of that half-width on either side. The SVM
picks the boundary that sits in the middle of the *widest* street.

In [ ]:
# Orient candidate lines using the two closest opposite-class points (pure geometry, no SVM yet).
A, B = X[y == 0], X[y == 1]
pair = np.unravel_index(
    np.argmin(np.linalg.norm(A[:, None, :] - B[None, :, :], axis=2)), (len(A), len(B))
)
mid = (A[pair[0]] + B[pair[1]]) / 2
seg = B[pair[1]] - A[pair[0]]
base = np.arctan2(seg[1], seg[0])  # the boundary's normal runs along this segment

xs = np.linspace(X[:, 0].min() - 0.6, X[:, 0].max() + 0.6, 200)
tilts = {"tilted by -18 deg": -18, "the widest street": 0, "tilted by +18 deg": 18}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
for ax, (name, deg) in zip(axes, tilts.items(), strict=True):
    u = np.array([np.cos(base + np.deg2rad(deg)), np.sin(base + np.deg2rad(deg))])
    c = -u @ mid
    margin = float(np.min(np.abs(X @ u + c)))  # distance to the nearest point
    for k in (0, 1):
        ax.scatter(
            *X[y == k].T, color=colors.CLASS_CYCLE[k],
            edgecolor=colors.COLORS["text"], linewidth=0.6, s=40,
        )
    ax.plot(xs, -(u[0] * xs + c) / u[1], color=colors.COLORS["text"], linewidth=1.6)
    for sgn in (-1, 1):
        ax.plot(
            xs, -(u[0] * xs + c) / u[1] + sgn * margin / u[1],
            color=colors.COLORS["muted"], linewidth=1.0, linestyle="--",
        )
    ax.set_title(f"{name}\nmargin = {margin:.3f}")
    ax.set_xlabel("feature 1 (standardized)")
    ax.set_ylim(X[:, 1].min() - 0.6, X[:, 1].max() + 0.6)
axes[0].set_ylabel("feature 2 (standardized)")
fig.tight_layout()
plt.show()

**Read the figure.** All three dashed-edged streets separate the two clouds — every point stays
on its own side. But their widths differ: tilting the boundary away from the centre narrows the
street, because the boundary then leans toward one cloud. The middle panel, with the boundary set
perpendicular to the line joining the two nearest opposite points, has the widest street of the three.
That width is exactly what the support vector machine maximizes. (For the two tilted panels the
boundary is no longer centred in the gap — it leans toward one cloud — so the band drawn around it
marks the distance to the *nearest* point, not an equal gap on both sides; that lean is why the margin
is smaller.) The next cells find that widest street by hand.

## The support vectors pin the street

Here is the by-hand recipe for the widest street. Find the **two closest opposite-class points**. The
widest possible street has its boundary running as the **perpendicular bisector** of the segment
joining them — equidistant from both — and its width is exactly the distance between them. Those two
points are the **support vectors**: they alone hold the boundary in place.

One honest caveat, stated up front: this clean two-point recipe is exact *here* because the widest
street happens to be pinned by a single closest pair. In general the boundary sits between the two
classes' nearest facing regions — the edges of their **convex hulls** — and **three or more** points
can land on the street's edges. You meet exactly two support vectors in this notebook; you meet the
messier case the moment the classes crowd together (the next notebook).

In [ ]:
# By hand: the two closest opposite-class points, and the bisector they define.
distances = np.linalg.norm(A[:, None, :] - B[None, :, :], axis=2)
i, j = np.unravel_index(np.argmin(distances), distances.shape)
sv_minus, sv_plus = A[i], B[j]            # the closest opposite-class pair
midpoint = (sv_minus + sv_plus) / 2       # the boundary passes through here
street_width = float(np.linalg.norm(sv_plus - sv_minus))
w_hand = sv_plus - sv_minus               # the boundary is perpendicular to this segment
b_hand = -w_hand @ midpoint

print(f"closest pair:  class-0 point {sv_minus.round(3)}   class-1 point {sv_plus.round(3)}")
print(f"street width = distance between them = {street_width:.4f}")
print(f"half-margin (boundary to each support vector) = {street_width / 2:.4f}")
print(f"boundary: through the midpoint {midpoint.round(3)}, perpendicular to that segment")

**Read the result.** Here, two points out of forty fix the entire boundary — in general a
handful, sometimes more than two, share that job. The midpoint between them lies on the street's centre
line, and the boundary runs perpendicular to the segment joining them. Every other point sits strictly
outside the street and has no say in where the boundary goes — a fact we will see directly in a
moment.

## Why the margin is $2 / \lVert w \rVert$

Write the boundary as $w \cdot x + b = 0$. Scaling both $w$ and $b$ by any positive number leaves that
line exactly where it is, so we may **choose** the scale freely — and we choose it so the closest
points satisfy $y\,(w \cdot x + b) = 1$ (the support vectors land on the $\pm 1$ lines). This names a
convention, not a new assumption about the data. With it, a short calculation gives the street width as

$$ \text{margin} = \frac{2}{\lVert w \rVert}. $$

So **maximizing the margin is the same as minimizing $\lVert w \rVert$** — a clean optimization
problem. Finding the optimum is the library's job (as gradient descent was in Chapter 03); our job is
to know what it is solving for. In code we approximate the hard margin with a very large penalty,
`C = 1e6`; the real `C` dial — what to do when no street separates the classes at all — is the next
notebook.

In [ ]:
# The library solves min ||w|| subject to keeping every point off the street. Does it match?
svc = SVC(kernel="linear", C=1e6).fit(X, y)  # C huge => a (near) hard margin
w, b = svc.coef_[0], svc.intercept_[0]
norm_w = float(np.linalg.norm(w))
cos_align = float(abs(w @ w_hand) / (norm_w * np.linalg.norm(w_hand)))
functional = [round(float((2 * y[k] - 1) * (w @ X[k] + b)), 3) for k in svc.support_]

print(f"SVM   ||w|| = {norm_w:.4f}    margin 2/||w|| = {2 / norm_w:.4f}")
print(f"by hand                       street width   = {street_width:.4f}")
print(f"support-vector indices: {np.sort(svc.support_)}    (n = {svc.support_.size})")
print(f"alignment cos(w_SVM, by-hand segment) = {cos_align:.4f}   (1.0 = same direction)")
print(f"midpoint on the SVM boundary?  w . m + b = {w @ midpoint + b:+.2e}")
print(f"functional margins y(w.x+b) at the support vectors: {functional}")

In [ ]:
fig = viz.plot_svm_decision(svc, X, y)
fig.axes[0].set_xlabel("feature 1 (standardized)")
fig.axes[0].set_ylabel("feature 2 (standardized)")
fig.axes[0].set_title("The maximum-margin street")
plt.show()

**Read the figure.** The solid line is the decision boundary; the two dashed lines are the
edges of the street, the $\pm 1$ level set of $w \cdot x + b$. The ringed points are the **support
vectors** — they sit exactly on the street's edges, which is why their functional margin printed as
$\pm 1$ above. The `SVC` found precisely the street we built by hand: same width ($2/\lVert w \rVert$
equals the distance between the two points), same boundary (the alignment cosine is $1.0$ and the
midpoint lies on the line).

## Only the support vectors matter

Because the street is pinned by those few points, the boundary is *sparse* in the data: it depends on
the support vectors and ignores everything else. Two quick experiments make this concrete — delete a
far-away point, and move a support vector.

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
grid_x = np.linspace(X[:, 0].min() - 0.6, X[:, 0].max() + 0.6, 200)


def boundary_y(weights, bias):
    return -(weights[0] * grid_x + bias) / weights[1]


# LEFT: delete a far-away non-support point, refit -> the boundary does not move.
non_sv = np.array([k for k in range(len(X)) if k not in svc.support_])
drop = int(non_sv[np.argmax(np.abs(X[non_sv] @ w + b))])  # the farthest non-support point
keep = np.array([k for k in range(len(X)) if k != drop])
svc_L = SVC(kernel="linear", C=1e6).fit(X[keep], y[keep])
for k in (0, 1):
    axL.scatter(*X[y == k].T, color=colors.CLASS_CYCLE[k],
                edgecolor=colors.COLORS["text"], linewidth=0.6, s=40)
axL.scatter(*X[drop], s=190, facecolors="none", edgecolors=colors.COLORS["error"],
            linewidths=2.0, label="deleted point")
axL.plot(grid_x, boundary_y(w, b), color=colors.COLORS["text"], linewidth=2.6, label="original")
axL.plot(grid_x, boundary_y(svc_L.coef_[0], svc_L.intercept_[0]),
         color=colors.COLORS["highlight"], linewidth=1.2, linestyle="--", label="refit without it")
axL.set_title("delete a far point -> boundary unchanged")
axL.set_xlabel("feature 1 (standardized)")
axL.set_ylabel("feature 2 (standardized)")
axL.legend(loc="best")

# RIGHT: move a support vector, refit -> the boundary follows it.
moved = X.copy()
sv0 = int(svc.support_[0])
moved[sv0] = moved[sv0] + np.array([1.4, -0.5])
svc_R = SVC(kernel="linear", C=1e6).fit(moved, y)
for k in (0, 1):
    axR.scatter(*moved[y == k].T, color=colors.CLASS_CYCLE[k],
                edgecolor=colors.COLORS["text"], linewidth=0.6, s=40)
axR.annotate("", xy=moved[sv0], xytext=X[sv0],
             arrowprops={"arrowstyle": "->", "color": colors.COLORS["error"], "lw": 1.8})
axR.plot(grid_x, boundary_y(w, b), color=colors.COLORS["text"], linewidth=1.2,
         linestyle="--", label="original")
axR.plot(grid_x, boundary_y(svc_R.coef_[0], svc_R.intercept_[0]),
         color=colors.COLORS["highlight"], linewidth=2.6, label="refit after the move")
axR.set_title("move a support vector -> boundary shifts")
axR.set_xlabel("feature 1 (standardized)")
axR.legend(loc="best")
fig.tight_layout()
plt.show()

**Read the figure.** On the left, deleting the farthest point and refitting leaves the boundary
exactly where it was — the two lines lie on top of each other. On the right, nudging a single support
vector drags the whole boundary along with it. The crowd of interior points has no vote; only the
support vectors hold the street up. This sparsity is part of why SVMs travel well to new data.

In [ ]:
# A separator that does NOT maximize the street: logistic regression (Chapter 03).
logreg = LogisticRegression().fit(X, y)
wl, bl = logreg.coef_[0], logreg.intercept_[0]
margin_lr = float(np.min(np.abs(X @ wl + bl) / np.linalg.norm(wl)))
margin_svm = float(np.min(np.abs(X @ w + b) / norm_w))
print(f"logistic regression: distance to the nearest point = {margin_lr:.4f}")
print(f"SVM (widest margin): distance to the nearest point = {margin_svm:.4f}")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for k in (0, 1):
    ax.scatter(*X[y == k].T, color=colors.CLASS_CYCLE[k],
               edgecolor=colors.COLORS["text"], linewidth=0.6, s=40, label=f"class {k}")
ax.plot(grid_x, boundary_y(w, b), color=colors.COLORS["highlight"], linewidth=2.0,
        label="SVM (widest margin)")
ax.plot(grid_x, boundary_y(wl, bl), color=colors.COLORS["model"], linewidth=2.0,
        linestyle="--", label="logistic regression")
ax.set_xlabel("feature 1 (standardized)")
ax.set_ylabel("feature 2 (standardized)")
ax.legend(loc="best")
plt.show()

**Read the figure.** Logistic regression separates the two clouds perfectly too — but its line
sits closer to the data, a narrower street ($0.77$ versus the SVM's $0.86$). Same data, a different
*criterion*: logistic regression placed its line by probability, the SVM by the widest margin. When the
classes overlap (the next notebook), this difference in *what* we optimize will matter even more.

## Your turn

1. **Easy — read three margins.** Looking at the three panels of the first figure, say which line the
   support vector machine prefers, and how the printed margins tell you.
2. **Medium — name the support vectors.** From the maximum-margin figure, identify which points are the
   support vectors and explain how you can tell (what is special about where they sit?).
3. **Harder — argue the invariance.** Using the deletion/move figure, explain why deleting the far
   point (left panel) left the boundary exactly where it was, while moving a support vector (right
   panel) changed everything. Argue from what you saw, not from any machinery from later notebooks.

## What you built

- You saw that on separable data **many** lines work, and that the SVM picks the one with the **widest
  margin** — the most defensible, most robust separator.
- You found that street **by hand**: a few boundary-touching points (the **support vectors**) pin it —
  here, the two closest opposite-class points — the boundary is their perpendicular bisector, and the
  width is the distance between them.
- You connected the geometry to the algebra: the margin is $2 / \lVert w \rVert$, so maximizing it
  means **minimizing $\lVert w \rVert$** — and a linear `SVC` found exactly the street you built.
- You watched the solution's **sparsity**: only the support vectors matter; deleting a far point
  changes nothing, moving a support vector changes everything.

**Vocabulary:** margin · the street · support vectors · maximum-margin boundary · $2/\lVert w \rVert$.

## Going further (optional)

The widest-margin boundary is the solution of a constrained optimization problem,

$$ \min_{w,\,b} \tfrac{1}{2}\lVert w \rVert^2 \quad\text{subject to}\quad y_i\,(w \cdot x_i + b) \ge 1
\ \text{ for every point } i. $$

Geometrically, the widest street runs between the two classes' **convex hulls** — the smallest convex
regions containing each class — which is the honest, general version of the closest-pair recipe: when
a hull edge (not a single point) faces the other class, three or more points can sit on the street's
edges (ESL §12.2).

Rewriting it brings out a striking fact: the solution depends on the data **only through dot products**
between points, $x_i \cdot x_j$, and only the support vectors carry weight. That dot-product-only view
is the door to everything ahead — letting some points sit inside the street (the **soft margin**, next
notebook) and replacing the dot product to bend the boundary (the **kernel trick**, NB 3). None of it
is needed to use what you built here.

## References

- Cortes C, Vapnik V (1995). *Support-vector networks.* Machine Learning 20:273-297.
  DOI: [10.1007/BF00994018](https://doi.org/10.1007/BF00994018)
- Boser BE, Guyon IM, Vapnik VN (1992). *A training algorithm for optimal margin classifiers.*
  Proc. COLT. DOI: [10.1145/130385.130401](https://doi.org/10.1145/130385.130401)
- Hastie T, Tibshirani R, Friedman J (2009). *The Elements of Statistical Learning*, §12.
  DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)
- James G, Witten D, Hastie T, Tibshirani R (2021). *An Introduction to Statistical Learning*, §9.
  DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1)

*Previous: Module 04 — Decision Trees · Next: 02 — The soft margin and the cost C*